# TopK / BottomK Spread Research Notebook

Validate the experimental LGBMRegressor + T+10 strategy with the current `src.core` pure-function interfaces.

The goal is to inspect scores, TopK/BottomK selection, rolling returns, and spread metrics before using `src.backtest_strategy.run_backtest()` for the full simulation.

In [ ]:
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print(f'Project root: {ROOT}')

In [ ]:
import pickle

import matplotlib.pyplot as plt
import pandas as pd

from src.core import (
    build_rolling_portfolio,
    compute_spread,
    generate_scores,
    select_bottomk,
    select_topk,
)

## 1. Load model, features, and close prices

In [ ]:
MODEL_PATH = ROOT / 'models' / 'us_regressor.pkl'
FEATURE_PATH = ROOT / 'data' / 'features_test.parquet'
PRICE_PATH = ROOT / 'data' / 'close_prices.parquet'
BENCHMARK = 'QQQ'

with MODEL_PATH.open('rb') as f:
    model = pickle.load(f)
feature_panel = pd.read_parquet(FEATURE_PATH)      # MultiIndex: datetime x instrument
close = pd.read_parquet(PRICE_PATH)                # Wide frame: datetime x instrument

print(type(model))
print(feature_panel.shape, close.shape)

## 2. Generate score panel

In [ ]:
score_series = generate_scores(model, feature_panel)
score_panel = score_series.to_frame('score')
score_series.describe()

## 3. Inspect one rebalance date

In [ ]:
date_level = 'datetime' if 'datetime' in score_series.index.names else 0
sample_date = score_series.index.get_level_values(date_level).unique()[-1]
daily_scores = score_series.xs(sample_date, level=date_level).dropna()

top10 = select_topk(daily_scores, k=10, guardrail=False)
bottom10 = select_bottomk(daily_scores, k=10)
print('Sample date:', sample_date)
print('Top10:', top10)
print('Bottom10:', bottom10)
daily_scores.hist(bins=50)
plt.title(f'Score distribution — {sample_date}')
plt.tight_layout()
plt.show()

## 4. Build return panel

In [ ]:
return_panel = close.pct_change().stack().rename('return').to_frame()
return_panel.index.names = ['datetime', 'instrument']

benchmark_returns = None
if BENCHMARK in close.columns:
    benchmark_returns = close[BENCHMARK].pct_change().rename('benchmark_return')
    return_panel = return_panel.drop(index=BENCHMARK, level='instrument', errors='ignore')
return_panel.head()

## 5. Rolling TopK / BottomK portfolio

In [ ]:
result = build_rolling_portfolio(
    score_panel=score_panel,
    return_panel=return_panel,
    k=10,
    holding_days=10,
    long_only=False,
    guardrail=True,
    benchmark_returns=benchmark_returns,
)
result.spread_equity.tail()

## 6. Spread metrics

In [ ]:
metrics = compute_spread(
    long_returns=result.long_returns,
    short_returns=result.short_returns,
    bench_returns=result.bench_returns if len(result.bench_returns) else None,
)
for key, value in metrics.items():
    if not isinstance(value, pd.Series):
        print(f'{key:<20}: {value:.4f}')
metrics['spread_equity'].plot(title='Long-short spread equity', figsize=(10, 4))
plt.tight_layout()
plt.show()

## 7. Full backtest handoff

In [ ]:
from src.backtest_strategy import run_backtest

# full = run_backtest(
#     start_date='2025-01-01',
#     end_date='2026-01-01',
#     topk=10,
#     bottomk=10,
#     holding_days=10,
#     model_path=str(MODEL_PATH),
# )
# full['spread_metrics']['spread_equity'].plot()